# Aufgabe 12.3: Regression – lineares Modell für `final_weight`

Dieses Notebook trainiert die drei linearen Regressionsmodelle aus der Aufgabenstellung, vergleicht die MSE-Werte und erstellt anschließend die Prognose-Datei `reg_Außerlechner-Kleemayr.csv`.


## 1. Bibliotheken importieren

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## 2. Einstellungen

Passe die Pfade nur dann an, wenn deine Dateien in einem anderen Ordner liegen.

In [5]:
TRAIN_DATA_PATH = "database/data.csv"
PREDICT_DATA_PATH = "database/X.csv"

OUTPUT_PATH = "regression/reg_Außerlechner-Kleemayr.csv"
RESULTS_PATH = "regression/regression_results.csv"

## 3. Trainingsdaten laden und Spaltennamen anpassen

In [6]:
df = pd.read_csv(TRAIN_DATA_PATH)

# Spaltennamen aus data.csv an die Namen aus X.csv / Angabe anpassen
df = df.rename(columns={
    "red_fill_level": "fill_level_grams_red",
    "blue_fill_level": "fill_level_grams_blue",
    "green_fill_level": "fill_level_grams_green",

    "red_vibration": "vibration_index_red",
    "blue_vibration": "vibration_index_blue",
    "green_vibration": "vibration_index_green",

    "red_temp": "temperature_red",
    "blue_temp": "temperature_blue",
    "green_temp": "temperature_green"
})

df.head()

,bottle,recipe,fill_level_grams_red,fill_level_grams_blue,fill_level_grams_green,vibration_index_red,vibration_index_blue,vibration_index_green,temperature_red,temperature_blue,temperature_green,final_weight,is_cracked,drop_oscillation
0,81740829,7,56.355195,546.578942,492.503147,92.356816,-1.370080,92.509509,34.178723,33.413129,32.952256,24.884694,0,"[""0.0000000000"", ""0.5244412309"", ""0.7645641276..."
1,81740831,7,48.103788,545.091611,484.488770,101.471201,2.225236,91.892157,34.802276,33.404674,36.729803,24.990835,0,"[""0.0000000000"", ""0.5229347540"", ""0.8407549269..."
2,81740833,7,40.798795,543.218725,477.113824,95.118647,11.365305,94.427851,33.417616,33.655806,32.392244,25.096081,0,"[""0.0000000000"", ""0.5759815506"", ""1.6015261906..."
3,81740835,7,33.256828,542.119956,469.731048,102.740527,-7.366184,90.383358,31.987952,31.871362,33.512363,24.527522,0,"[""0.0000000000"", ""0.3562968453"", ""0.6045360191..."
4,81740837,7,26.283345,540.675043,462.840897,89.275574,-1.646569,80.654556,32.490402,35.946574,33.709014,22.961452,0,"[""0.0000000000"", ""0.0048046704"", ""0.0081833030..."


## 4. Feature-Sets laut Angabe definieren

In [7]:
target = "final_weight"

feature_sets = [
    [
        "fill_level_grams_red",
        "fill_level_grams_blue",
        "fill_level_grams_green"
    ],
    [
        "fill_level_grams_red",
        "fill_level_grams_blue",
        "fill_level_grams_green",
        "vibration_index_red",
        "vibration_index_blue",
        "vibration_index_green"
    ],
    [
        "fill_level_grams_red",
        "fill_level_grams_blue",
        "fill_level_grams_green",
        "vibration_index_red",
        "vibration_index_blue",
        "vibration_index_green",
        "temperature_red",
        "temperature_blue",
        "temperature_green"
    ]
]

## 5. Modelle trainieren und MSE berechnen

In [8]:
results = []
models = []

for features in feature_sets:
    X = df[features]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mse_train = mean_squared_error(y_train, y_train_pred)
    mse_test = mean_squared_error(y_test, y_test_pred)

    results.append({
        "Genutzte Spalten (X)": ", ".join(features),
        "Modell-Typ": "Linear",
        "MSE Training": mse_train,
        "MSE Test": mse_test
    })

    models.append({
        "features": features,
        "model": model,
        "mse_train": mse_train,
        "mse_test": mse_test
    })

## 6. Ergebnis-Tabelle anzeigen und speichern

In [14]:
results_df = pd.DataFrame(results)

results_df.to_csv(RESULTS_PATH, index=False)
results_df

,Genutzte Spalten (X),Modell-Typ,MSE Training,MSE Test
0,"fill_level_grams_red, fill_level_grams_blue, f...",Linear,11.288824,16.015207
1,"fill_level_grams_red, fill_level_grams_blue, f...",Linear,0.120408,0.141158
2,"fill_level_grams_red, fill_level_grams_blue, f...",Linear,0.119477,0.139538


## 7. Bestes Modell auswählen

In [10]:
best = min(models, key=lambda x: x["mse_test"])

best_model = best["model"]
best_features = best["features"]

print("Bestes Modell:")
print(best_features)
print(f"MSE Training: {best['mse_train']:.6f}")
print(f"MSE Test:     {best['mse_test']:.6f}")

Bestes Modell:
['fill_level_grams_red', 'fill_level_grams_blue', 'fill_level_grams_green', 'vibration_index_red', 'vibration_index_blue', 'vibration_index_green', 'temperature_red', 'temperature_blue', 'temperature_green']
MSE Training: 0.119477
MSE Test:     0.139538


## 8. Formel des besten Modells ausgeben

In [11]:
formula_parts = []

for coef, feature in zip(best_model.coef_, best_features):
    formula_parts.append(f"({coef:.6f} * {feature})")

formula = "y = " + " + ".join(formula_parts) + f" + {best_model.intercept_:.6f}"

print("Formel des besten Modells:")
print(formula)

Formel des besten Modells:
y = (0.000366 * fill_level_grams_red) + (0.000230 * fill_level_grams_blue) + (0.000291 * fill_level_grams_green) + (0.098992 * vibration_index_red) + (0.075775 * vibration_index_blue) + (0.099542 * vibration_index_green) + (-0.007582 * temperature_red) + (0.031290 * temperature_blue) + (0.000579 * temperature_green) + 4.542869


## 9. X.csv laden und Prognose erstellen

In [12]:
X_pred = pd.read_csv(PREDICT_DATA_PATH)

missing_columns = [col for col in best_features if col not in X_pred.columns]

if missing_columns:
    raise ValueError(f"Diese Spalten fehlen in X.csv: {missing_columns}")

y_hat = best_model.predict(X_pred[best_features])

## 10. Abgabe-Datei speichern

In [15]:
reg_df = pd.DataFrame({
    "Flaschen ID": X_pred["bottle"],
    "y_hat": y_hat
})

reg_df.to_csv(OUTPUT_PATH, index=False)

print(f"Prognose wurde gespeichert unter: {OUTPUT_PATH}")
reg_df.head()

Prognose wurde gespeichert unter: regression/reg_Außerlechner-Kleemayr.csv


,Flaschen ID,y_hat
0,368,51.371767
1,369,49.215247
2,370,49.710928
3,371,26.581392
4,372,25.574183
